# Optimering av busslinjenät för Åland med genetisk algoritm

Detta är Åland-motsvarigheten till Mandl-notebooken (`Genetic_Algorithm_UTRP/Mandl Python/Working Mandl.ipynb`).
En **genetisk algoritm (GA)** söker en uppsättning busslinjer som **minimerar passagerarnas upplevda restid**
under eftermiddagens rusningstrafik (EM).

## Indata
1. **Hållplatser** – koordinater (GeoJSON, 602 plattformar).
2. **Centroider** – zonernas koordinater (`centroid_nyckel.csv`, 1477 zoner).
3. **OD-matris** – efterfrågan zon→zon för EM (`OD_matris.xlsx`, 1477×1477, gles).

## Parametrar du själv sätter (se cellen *Parametrar*)
- Antal busslinjer.
- Tillåtet längdspann per linje (km).
- En central hållplats som **alla** linjer måste passera.
- Turtäthet (headway), som väntetiden beräknas schablonmässigt ifrån.

## Optimeringskriterier (upplevd restid)
- **Åktid** mellan hållplatser hämtas från **OSRM** (bil-profil) och cachas i CSV.
- **Byte** straffas med **+5 minuter**.
- **Väntetid** räknas schablonmässigt som `headway / 2` och varje väntad minut väger **×2** upplevda minuter
  (t.ex. 10 min headway → 5 min väntetid → 10 upplevda minuter). Gäller varje påstigning (start + varje byte).

> **Restidscache:** Före varje körning kontrolleras att restider finns för *alla* nuvarande hållplatser.
> Saknas någon (t.ex. efter att du uppdaterat hållplatslistan) beräknas matrisen om automatiskt.

## Körning i Google Colab (valfritt)

Kör cellen nedan **först** om du använder Google Colab. Den monterar din Google Drive och klonar (eller
uppdaterar) repot dit, så att data och resultat finns kvar mellan sessioner. Repot är **publikt – ingen
nyckel/token behövs**. Utanför Colab gör cellen ingenting.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/JesperHelen/-land.git'   # publikt repo – ingen nyckel behövs
    REPO_DIR = '/content/drive/MyDrive/aland_ga'            # ändra om du vill lägga repot någon annanstans

    if os.path.isdir(os.path.join(REPO_DIR, '.git')):
        print('Uppdaterar befintlig klon i din Drive ...')
        os.system(f'git -C "{REPO_DIR}" pull --ff-only')
    else:
        print('Klonar repot till din Drive (kan ta en stund) ...')
        os.system(f'git clone "{REPO_URL}" "{REPO_DIR}"')

    os.chdir(REPO_DIR)
    print('Arbetskatalog:', os.getcwd())
    assert os.path.exists(os.path.join('data', 'hallplatser.geojson')), \
        'Hittade inte data/hallplatser.geojson – kontrollera REPO_URL/REPO_DIR ovan.'
    print('Klart – data hittad. Kör vidare nedan.')
else:
    print('Inte i Google Colab – hoppar över detta steg (kör notebooken lokalt som vanligt).')

## 0. Beroenden
Installera vid behov med `pip install -r requirements.txt`.

In [ ]:
import os, json, math, time, random
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra

try:
    from tqdm.auto import tqdm
except Exception:               # tqdm är valfritt
    def tqdm(x, **k):
        return x

## 1. Parametrar
Ändra dessa värden inför en körning. Sökvägarna utgår från att notebooken ligger i `-land/notebooks/`
och att indatafilerna ligger i `-land/data/`.

In [ ]:
from pathlib import Path

# --- Hitta repo-mappen automatiskt (fungerar oavsett var kärnan startas ifrån) ---
# Vi letar efter mappen som innehåller 'data/hallplatser.geojson'. Notebooken kan köras från notebooks/,
# från repo-roten (-land/), en nivå ovanför, ELLER från t.ex. Google Colab där repot klonats till en
# undermapp som /content/-land/ (då söker vi även i undermappar).
def hitta_basmapp():
    mal = Path('data') / 'hallplatser.geojson'
    def traff(bas):
        try:
            return (bas / mal).exists()
        except Exception:
            return False
    # 1) snabba kandidater: cwd, förälder, farförälder
    for bas in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if traff(bas):
            return bas.resolve()
    # 2) undermappar (djup 1–2) under cwd och dess förälder – fångar t.ex. /content/-land/
    for rot in [Path.cwd(), Path.cwd().parent]:
        try:
            for barn in rot.iterdir():
                if barn.is_dir():
                    if traff(barn):
                        return barn.resolve()
                    try:
                        for barnbarn in barn.iterdir():
                            if barnbarn.is_dir() and traff(barnbarn):
                                return barnbarn.resolve()
                    except Exception:
                        pass
        except Exception:
            pass
    # 3) gå uppåt i katalogträdet
    p = Path.cwd().resolve()
    for _ in range(8):
        if traff(p):
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd().resolve()                 # sista utväg (ger tydligt fel + tips nedan)

# Sätt BAS_DIR manuellt här om din datamapp ligger på en annan plats. Exempel:
#   BAS_DIR = Path('/content/-land')          # Google Colab efter: !git clone .../-land
#   BAS_DIR = Path('/content/drive/MyDrive/-land')   # om repot ligger på Google Drive
BAS_DIR = hitta_basmapp()
if not (BAS_DIR / 'data' / 'hallplatser.geojson').exists():
    print('VARNING: hittade inte data/hallplatser.geojson (utgick från', str(BAS_DIR) + ').')
    print('  Sätt BAS_DIR manuellt ovan till mappen som innehåller data/ (repo-roten "-land").')
    print('  Tips: kör i en cell   !find / -name hallplatser.geojson 2>/dev/null   för att hitta sökvägen.')
else:
    print('Basmapp (repo):', BAS_DIR)

# --- Sökvägar ---
DATA_DIR       = str(BAS_DIR / 'data')
OUTPUT_DIR     = str(BAS_DIR / 'output')
GEOJSON_HPL    = os.path.join(DATA_DIR, 'hallplatser.geojson')
CSV_CENTROIDER = os.path.join(DATA_DIR, 'centroid_nyckel.csv')
XLSX_OD        = os.path.join(DATA_DIR, 'OD_matris.xlsx')
CSV_OD_CACHE   = os.path.join(DATA_DIR, 'od_long.csv')            # gles OD-cache (skapas)
CSV_RESTIDER   = os.path.join(OUTPUT_DIR, 'restider_hallplatser.csv')   # OSRM-cache (skapas)
CSV_RESULTAT   = os.path.join(OUTPUT_DIR, 'basta_linjenat.csv')   # resultat (skapas)
PNG_KARTA      = os.path.join(OUTPUT_DIR, 'linjenat_karta.png')

# --- Nätverksparametrar ---
ANTAL_LINJER        = 6            # antal busslinjer (din parameter)
LINJE_LANGD_MIN_KM  = 6.0          # tillåtet längdspann per linje (km)
LINJE_LANGD_MAX_KM  = 40.0
CENTRAL_NAMN        = 'Bussplan'                 # central hållplats – matchas mot namn
CENTRAL_KOORD       = (19.9422816, 60.1022269)   # reserv om namnet inte hittas (Mariehamn, Bussplan)

# --- Restidsmodell ---
HEADWAY_MIN         = 30           # turtäthet (min). Skalär => samma för alla linjer.
                                   # Ange en lista med längd ANTAL_LINJER för headway per linje.
BYTESSTRAFF_MIN     = 5.0          # +5 min per byte
VANTETIDSVIKT       = 2.0          # varje väntad minut = 2 upplevda minuter
STRAFF_OBETJANAD    = 120.0        # straff (upplevda min) per enhet efterfrågan som saknar förbindelse

# --- OSRM ---
ANVAND_OSRM         = True
OSRM_BASE_URL       = 'https://router.project-osrm.org'   # egen server: t.ex. 'http://localhost:5000'
OSRM_MAX_TABLE      = 100          # max antal koordinater per /table-anrop (publik demo = 100)
FALLBACK_HASTIGHET  = 45.0         # km/h – används bara om OSRM inte kan nås

# --- Aggregering & GA ---
KLUSTER_M           = 40.0         # slå ihop hållplatser inom detta avstånd (m); 0 => ingen hopslagning
ANTAL_GRANNAR       = 10           # antal närmaste grannar som linjer kan gå vidare till
POP_STORLEK         = 40
GENERATIONER        = 90
MUTATIONSGRAD       = 0.35
SLUMPFRO            = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)
rng = random.Random(SLUMPFRO)
np.random.seed(SLUMPFRO)
print('Parametrar laddade.')

## 2. Geografiska hjälpfunktioner

In [ ]:
JORDRADIE_M = 6371000.0  # jordens medelradie (m)

def haversine_m(lon1, lat1, lon2, lat2):
    la1, la2 = math.radians(lat1), math.radians(lat2)
    dla = math.radians(lat2 - lat1); dlo = math.radians(lon2 - lon1)
    h = math.sin(dla/2)**2 + math.cos(la1)*math.cos(la2)*math.sin(dlo/2)**2
    return 2*JORDRADIE_M*math.asin(math.sqrt(h))

def haversine_matris_m(lons, lats):
    """Vektoriserad NxN haversine (meter)."""
    lons = np.radians(np.asarray(lons, float)); lats = np.radians(np.asarray(lats, float))
    dlon = lons[None, :] - lons[:, None]
    dlat = lats[None, :] - lats[:, None]
    a = np.sin(dlat/2)**2 + np.cos(lats)[:, None]*np.cos(lats)[None, :]*np.sin(dlon/2)**2
    return 2*JORDRADIE_M*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

## 3. Inläsning av indata
Hållplatser läses från GeoJSON, centroider från CSV och OD-matrisen från Excel. OD-matrisen är stor
(1477×1477) men gles, så den läses en gång till gles long-form `[origin_zon, dest_zon, efterfragan]` och cachas.

In [ ]:
def las_hallplatser(path):
    d = json.load(open(path, encoding='utf-8'))
    rader = []
    for f in d['features']:
        lon, lat = f['geometry']['coordinates']
        p = f['properties']
        namn = p.get('name') or p.get('name:sv') or p.get('description') or 'Namnlös'
        pid = p.get('@id') or f.get('id')
        rader.append({'platform_id': pid, 'namn': namn, 'lon': float(lon), 'lat': float(lat)})
    return pd.DataFrame(rader)

def las_centroider(path):
    df = pd.read_csv(path)
    df = df.rename(columns={'zon_id': 'zon_id', 'lon': 'lon', 'lat': 'lat'})
    return df

def las_od_matris(xlsx_path, cache_csv):
    """Läser OD-matrisen (1477x1477) till gles long-form och cachar."""
    if os.path.exists(cache_csv):
        return pd.read_csv(cache_csv)
    import openpyxl
    wb = openpyxl.load_workbook(xlsx_path, read_only=True, data_only=True)
    ws = wb['OD_matris']
    rader = ws.iter_rows(values_only=True)
    header = next(rader)
    dest_ids = list(header[1:])
    poster = []
    for row in rader:
        origin = row[0]
        if origin is None:
            continue
        for j, val in enumerate(row[1:]):
            if val:  # hoppa None och 0
                poster.append((origin, dest_ids[j], float(val)))
    df = pd.DataFrame(poster, columns=['origin_zon', 'dest_zon', 'efterfragan'])
    df.to_csv(cache_csv, index=False)
    return df

# ------------------------------------------------

In [ ]:
hallplatser = las_hallplatser(GEOJSON_HPL)
centroider  = las_centroider(CSV_CENTROIDER)
od_long     = las_od_matris(XLSX_OD, CSV_OD_CACHE)

print(f'Hållplatser (plattformar): {len(hallplatser)}')
print(f'Centroider (zoner):        {len(centroider)}')
print(f'OD-par (nollskilda):       {len(od_long)}   total efterfrågan: {od_long["efterfragan"].sum():.2f}')
print(f'Ursprungszoner med efterfrågan:   {od_long["origin_zon"].nunique()}')
print(f'Destinationszoner med efterfrågan:{od_long["dest_zon"].nunique()}')
hallplatser.head()

## 4. Hållplatsaggregering
Många hållplatser är *riktningspar* (samma läge, båda färdriktningarna). Vi slår ihop hållplatser inom
`KLUSTER_M` meter till en nod. Det minskar nätverket och ger tydligare linjer. Sätt `KLUSTER_M = 0`
för att behålla alla plattformar.

In [ ]:
def klustra_hallplatser(stops_df, cluster_m):
    """Slår ihop hållplatser inom cluster_m meter (riktningspar). union-find."""
    n = len(stops_df)
    lons = stops_df['lon'].to_numpy(); lats = stops_df['lat'].to_numpy()
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    # grov filtrering via grad-tröskel, exakt haversine per kandidatpar
    grader = cluster_m / 111320.0
    order = np.argsort(lats)
    for ii in range(n):
        i = order[ii]
        for jj in range(ii+1, n):
            j = order[jj]
            if lats[j] - lats[i] > grader:  # sorterad på lat -> kan bryta
                break
            if abs(lons[j]-lons[i]) > grader/max(math.cos(math.radians(lats[i])), 1e-6):
                continue
            if haversine_m(lons[i], lats[i], lons[j], lats[j]) <= cluster_m:
                union(i, j)
    grupper = {}
    for i in range(n):
        grupper.setdefault(find(i), []).append(i)
    kluster = []
    platform2kluster = {}
    for k, (root, idxs) in enumerate(sorted(grupper.items())):
        kid = f'H{k+1:04d}'
        sub = stops_df.iloc[idxs]
        # representativ koordinat = medel; namn = vanligast förekommande icke-namnlös
        namnrakning = sub['namn'][sub['namn'] != 'Namnlös']
        namn = namnrakning.mode().iloc[0] if len(namnrakning) else sub['namn'].iloc[0]
        kluster.append({'hpl_id': kid, 'namn': namn,
                        'lon': float(sub['lon'].mean()), 'lat': float(sub['lat'].mean()),
                        'antal_plattformar': len(idxs)})
        for pid in sub['platform_id']:
            platform2kluster[pid] = kid
    return pd.DataFrame(kluster), platform2kluster

# ------------------------------------------------

In [ ]:
if KLUSTER_M and KLUSTER_M > 0:
    hpl, platform2kluster = klustra_hallplatser(hallplatser, KLUSTER_M)
else:
    hpl = hallplatser.rename(columns={'platform_id': 'hpl_id'}).copy()
    hpl['antal_plattformar'] = 1
    platform2kluster = {p: p for p in hallplatser['platform_id']}

hpl = hpl.reset_index(drop=True)
N = len(hpl)
hpl_index = {h: i for i, h in enumerate(hpl['hpl_id'])}     # hpl_id -> radindex
print(f'Antal hållplatser efter aggregering: {N}')
hpl.head()

### Central hållplats
Den hållplats som **alla** linjer måste passera. Väljs i första hand på namn (`CENTRAL_NAMN`),
annars den hållplats som ligger närmast `CENTRAL_KOORD`.

In [ ]:
def hitta_central(hpl_df, namn, koord):
    traff = hpl_df[hpl_df['namn'].str.contains(namn, case=False, na=False)]
    if len(traff):
        # välj den kandidat som har flest plattformar (troligast huvudhållplatsen)
        return int(traff.sort_values('antal_plattformar', ascending=False).index[0])
    avst = haversine_vekt(koord[0], koord[1], hpl_df['lon'].to_numpy(), hpl_df['lat'].to_numpy())
    return int(np.argmin(avst))

central = hitta_central(hpl, CENTRAL_NAMN, CENTRAL_KOORD)
print(f'Central hållplats: idx={central}  id={hpl.loc[central, "hpl_id"]}  '
      f'namn="{hpl.loc[central, "namn"]}"  ({hpl.loc[central, "lon"]:.5f}, {hpl.loc[central, "lat"]:.5f})')

## 5. Restidsmatris via OSRM (med cache)
Åktiden mellan alla hållplatspar hämtas från OSRM (`/table`, bil-profil) och sparas i tydligt long-format
`from_id,to_id,duration_s,distance_m`. **Kontroll före körning:** finns en komplett cache för de aktuella
hållplatserna återanvänds den; saknas någon hållplats beräknas matrisen om automatiskt.

Om OSRM inte kan nås (t.ex. ingen internet/server) används en enkel **fallback** (fågelvägen × omvägsfaktor
delat på antagen hastighet) så att notebooken ändå kan köras – byt till en riktig OSRM-server för skarpa resultat.

In [ ]:
def bygg_restidsmatris_fallback(hpl_df, hastighet_kmh=40.0, omvag=1.3):
    lons = hpl_df['lon'].to_numpy(); lats = hpl_df['lat'].to_numpy()
    dist_km = haversine_matris_m(lons, lats)/1000.0*omvag
    dur_min = dist_km/hastighet_kmh*60.0
    np.fill_diagonal(dur_min, 0.0); np.fill_diagonal(dist_km, 0.0)
    return dur_min, dist_km

def _osrm_table_block(base, coords, src_idx, dst_idx, timeout=180):
    coord_str = ';'.join(f'{lo:.6f},{la:.6f}' for lo, la in coords)
    params = {'annotations': 'duration,distance',
              'sources': ';'.join(map(str, src_idx)),
              'destinations': ';'.join(map(str, dst_idx))}
    r = requests.get(f'{base}/table/v1/driving/{coord_str}', params=params, timeout=timeout)
    r.raise_for_status(); j = r.json()
    if j.get('code') != 'Ok':
        raise RuntimeError(f"OSRM: {j.get('code')} {j.get('message')}")
    return np.array(j['durations'], float), np.array(j['distances'], float)

def bygg_restidsmatris_osrm(hpl_df, base, max_table=100, paus=1.0, retries=4):
    coords = list(zip(hpl_df['lon'], hpl_df['lat'])); n = len(coords)
    b = max(1, max_table//2)
    DUR = np.full((n, n), np.nan); DIS = np.full((n, n), np.nan)
    blocks = [list(range(i, min(i+b, n))) for i in range(0, n, b)]
    for Si in blocks:
        for Dj in blocks:
            cc = [coords[k] for k in Si] + [coords[k] for k in Dj]
            si = list(range(len(Si))); di = list(range(len(Si), len(Si)+len(Dj)))
            for attempt in range(retries):
                try:
                    dur, dis = _osrm_table_block(base, cc, si, di); break
                except Exception:
                    if attempt == retries-1: raise
                    time.sleep(paus*(2**attempt))
            DUR[np.ix_(Si, Dj)] = dur; DIS[np.ix_(Si, Dj)] = dis
            time.sleep(paus)
    return DUR/60.0, DIS/1000.0

def spara_restider_csv(path, hpl_ids, dur_min, dist_km):
    n = len(hpl_ids); ids = np.array(hpl_ids)
    ii, jj = np.meshgrid(range(n), range(n), indexing='ij')
    pd.DataFrame({'from_id': ids[ii.ravel()], 'to_id': ids[jj.ravel()],
                  'duration_s': (dur_min*60).ravel(), 'distance_m': (dist_km*1000).ravel()}
                 ).to_csv(path, index=False)

def ladda_restider_csv(path, hpl_ids):
    """Returnerar (dur_min, dist_km) om cachen täcker ALLA hpl_ids, annars None."""
    if not os.path.exists(path): return None
    df = pd.read_csv(path)
    nuvarande = set(hpl_ids)
    if not nuvarande.issubset(set(df['from_id']) | set(df['to_id'])):
        return None
    idx = {h: i for i, h in enumerate(hpl_ids)}; n = len(hpl_ids)
    dur = np.full((n, n), np.nan); dis = np.full((n, n), np.nan)
    for f, t, ds, dm in df[['from_id', 'to_id', 'duration_s', 'distance_m']].itertuples(index=False):
        if f in idx and t in idx:
            dur[idx[f], idx[t]] = ds/60.0; dis[idx[f], idx[t]] = dm/1000.0
    if np.isnan(dur).any(): return None
    return dur, dis

def hamta_restider(hpl_df, csv_path, anvand_osrm=True, base=None, max_table=100):
    """Kontroll-först: använd cache om komplett, annars beräkna om automatiskt."""
    hpl_ids = hpl_df['hpl_id'].tolist()
    laddat = ladda_restider_csv(csv_path, hpl_ids)
    if laddat is not None:
        print(f'Restider: cache OK ({len(hpl_ids)} hållplatser).')
        return laddat
    print('Restider: cache saknas/ofullständig -> beräknar på nytt.')
    if anvand_osrm and base:
        try:
            dur, dis = bygg_restidsmatris_osrm(hpl_df, base, max_table=max_table)
            if np.isnan(dur).any():
                print('  Varning: OSRM gav luckor -> fyller med fallback.')
                fdur, fdis = bygg_restidsmatris_fallback(hpl_df)
                m = np.isnan(dur); dur[m] = fdur[m]; dis[m] = fdis[m]
        except Exception as e:
            print(f'  OSRM misslyckades ({e}) -> fallback (haversine).')
            dur, dis = bygg_restidsmatris_fallback(hpl_df)
    else:
        print('  Använder fallback-matris (haversine).')
        dur, dis = bygg_restidsmatris_fallback(hpl_df)
    spara_restider_csv(csv_path, hpl_ids, dur, dis)
    return dur, dis

# ------------------------------------------------

In [ ]:
dur_min, dist_km = hamta_restider(hpl, CSV_RESTIDER,
                                  anvand_osrm=ANVAND_OSRM, base=OSRM_BASE_URL, max_table=OSRM_MAX_TABLE)
print(f'Restidsmatris: {dur_min.shape}   '
      f'median åktid = {np.median(dur_min[dur_min>0]):.1f} min   max = {dur_min.max():.1f} min')

## 6. Koppla efterfrågan till hållplatser
Varje zon knyts till sin närmaste hållplats. Zon-OD:n aggregeras därmed till en **hållplats-OD-matris**
(gles), som GA:n väger restiderna mot.

In [ ]:
def koppla_zoner_till_hpl(centroider, hpl_df):
    hlon = hpl_df['lon'].to_numpy(); hlat = hpl_df['lat'].to_numpy()
    ids = hpl_df['hpl_id'].to_numpy()
    res = {}
    for _, z in centroider.iterrows():
        d = haversine_vekt(z['lon'], z['lat'], hlon, hlat)
        res[z['zon_id']] = ids[int(np.argmin(d))]
    return res

def haversine_vekt(lon, lat, lons, lats):
    la1 = math.radians(lat); la2 = np.radians(lats)
    dla = la2 - la1; dlo = np.radians(lons - lon)
    a = np.sin(dla/2)**2 + math.cos(la1)*np.cos(la2)*np.sin(dlo/2)**2
    return 2*JORDRADIE_M*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

def bygg_hpl_od(od_long, zon2hpl, hpl_index):
    """Aggregerar zon-OD till hållplats-OD (index-baserat)."""
    o = od_long['origin_zon'].map(zon2hpl).map(hpl_index)
    d = od_long['dest_zon'].map(zon2hpl).map(hpl_index)
    w = od_long['efterfragan'].to_numpy()
    mask = o.notna() & d.notna()
    o = o[mask].to_numpy().astype(int); d = d[mask].to_numpy().astype(int); w = w[mask.to_numpy()]
    df = pd.DataFrame({'o': o, 'd': d, 'w': w}).groupby(['o', 'd'], as_index=False)['w'].sum()
    return df

# ------------------------------------------------

In [ ]:
zon2hpl = koppla_zoner_till_hpl(centroider, hpl)
hpl_od  = bygg_hpl_od(od_long, zon2hpl, hpl_index)

print(f'Hållplats-OD-par: {len(hpl_od)}   total efterfrågan: {hpl_od["w"].sum():.2f}')
print(f'Unika ursprungshållplatser: {hpl_od["o"].nunique()}')

## 7. Genetisk algoritm

**Representation.** En individ = en lista av `ANTAL_LINJER` linjer. Varje linje är en sekvens av hållplatser
som **innehåller centralen** och vars längd ligger i `[LINJE_LANGD_MIN_KM, LINJE_LANGD_MAX_KM]`.

**Linjebygge.** Varje linje byggs som två armar ut från centralen mot var sitt *efterfrågeankare*
(en hållplats slumpad vägt efter `efterfrågan × avstånd från centralen`), genom att stega till närliggande
hållplatser i målets riktning. Viktningen får armarna att sträva utåt så att linjerna blir radiella och når
resenärer längre bort – likt ett verkligt Ålandsnät med nav i Mariehamn.

**Målfunktion (minimeras).** Upplevd restid för en resa =
`åktid (OSRM) + Σ byten·5 min + Σ påstigningar·(2 · headway/2)`.
Detta modelleras i en **linjegraf** där varje nod är `(hållplats, linje)`:
- åkkant inom en linje = OSRM-åktid,
- påstigningskant från resenärens start = upplevd väntetid för linjen,
- byteskant vid gemensam hållplats = `5 + upplevd väntetid för den nya linjen`.

Kortaste vägen beräknas med Dijkstra (scipy, multi-source: en sökning per ursprungshållplats). Objektivet är
efterfrågeviktad medelrestid plus ett straff för efterfrågan som saknar förbindelse.

In [ ]:
def bygg_grannar(dur_min, k):
    n = dur_min.shape[0]; g = []
    for i in range(n):
        d = dur_min[i].copy(); d[i] = np.inf
        g.append([int(x) for x in np.argsort(d)[:k]])
    return g

def linjelangd(line, dist_km):
    return sum(dist_km[line[i], line[i+1]] for i in range(len(line)-1))

def demand_per_hpl(hpl_od, N):
    """Total efterfrågan (in + ut) per hållplats — används för att styra linjer mot resenärer."""
    w = np.zeros(N)
    np.add.at(w, hpl_od['o'].to_numpy(), hpl_od['w'].to_numpy())
    np.add.at(w, hpl_od['d'].to_numpy(), hpl_od['w'].to_numpy())
    return w

def valj_ankare(hpl_vikt, rng):
    """Slumpar en 'ankarhållplats' vägt efter efterfrågan (dit en linjearm ska sträva)."""
    tot = hpl_vikt.sum()
    if tot <= 0:
        return rng.randrange(len(hpl_vikt))
    r = rng.random()*tot; ack = 0.0
    for i, v in enumerate(hpl_vikt):
        ack += v
        if ack >= r:
            return i
    return len(hpl_vikt)-1

def _vandra_mot(start, target, dur_min, dist_km, grannar, visited, budget_km, rng):
    """Girig geografisk vandring från start mot target längs närliggande hållplatser."""
    arm = []; cur = start; length = 0.0
    while length < budget_km and cur != target:
        cand = [s for s in grannar[cur] if s not in visited and s != cur]
        if not cand:
            break
        cand.sort(key=lambda s: dur_min[s, target])  # välj granne närmast målet
        nxt = cand[0] if rng.random() < 0.75 else rng.choice(cand[:min(3, len(cand))])
        length += dist_km[cur, nxt]; arm.append(nxt); visited.add(nxt); cur = nxt
    return arm

def bygg_linje(central, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng):
    """Bygger en linje som två armar från centralen mot var sitt efterfrågeankare."""
    target_len = rng.uniform(lmin, lmax)
    visited = {central}
    a1 = valj_ankare(hpl_vikt, rng); a2 = valj_ankare(hpl_vikt, rng)
    arm1 = _vandra_mot(central, a1, dur_min, dist_km, grannar, visited, target_len/2, rng)
    langd1 = sum(dist_km[central if i == 0 else arm1[i-1], arm1[i]] for i in range(len(arm1)))
    arm2 = _vandra_mot(central, a2, dur_min, dist_km, grannar, visited, target_len-langd1, rng)
    line = list(reversed(arm1)) + [central] + arm2
    if len(line) < 2:  # nödfall: minst en granne
        cand = [s for s in grannar[central] if s != central]
        if cand:
            line = [central, cand[0]]
    return laga_langd(line, central, lmin, lmax, dist_km)

def laga_langd(line, central, lmin, lmax, dist_km):
    line = list(line)
    while linjelangd(line, dist_km) > lmax and len(line) > 2:
        if line[0] != central:
            line = line[1:]
        elif line[-1] != central:
            line = line[:-1]
        else:
            break
    return line

def bygg_individ(central, antal_linjer, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng):
    return [bygg_linje(central, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng)
            for _ in range(antal_linjer)]

def mutera_linje(line, central, lmin, lmax, dist_km, grannar, rng):
    line = list(line)
    action = rng.choice(['lagg_v', 'lagg_h', 'ta_v', 'ta_h'])
    if action == 'lagg_v':
        cand = [s for s in grannar[line[0]] if s not in line]
        if cand: line.insert(0, rng.choice(cand))
    elif action == 'lagg_h':
        cand = [s for s in grannar[line[-1]] if s not in line]
        if cand: line.append(rng.choice(cand))
    elif action == 'ta_v' and len(line) > 2 and line[0] != central:
        line = line[1:]
    elif action == 'ta_h' and len(line) > 2 and line[-1] != central:
        line = line[:-1]
    return laga_langd(line, central, lmin, lmax, dist_km)

In [ ]:
class Utvarderare:
    """Bygger linjegraf och beräknar upplevd restid via scipy-Dijkstra (multi-source)."""
    def __init__(self, N, dur_min, hpl_od, headways_min, transfer_penalty, wait_weight, penalty_unserved):
        self.N = N; self.dur = dur_min
        self.transfer_penalty = transfer_penalty; self.penalty_unserved = penalty_unserved
        self.wait_weight = wait_weight
        self.headways = headways_min  # lista per linje (min)
        self.o_arr = hpl_od['o'].to_numpy(); self.d_arr = hpl_od['d'].to_numpy()
        self.w_arr = hpl_od['w'].to_numpy(); self.W = self.w_arr.sum()
        self.origin_stops = sorted(set(self.o_arr.tolist()))
        origin_row = {s: i for i, s in enumerate(self.origin_stops)}
        self.o_rows = np.array([origin_row[o] for o in self.o_arr])
        self.same = (self.o_arr == self.d_arr)

    def perc_wait(self, l):
        return self.wait_weight * (self.headways[l]/2.0)

    def _bygg_graf(self, routes):
        node_id = {}; stop_nodes = defaultdict(list); stops_here = defaultdict(list)
        for l, line in enumerate(routes):
            for s in line:
                nid = len(node_id); node_id[(s, l)] = nid
                stop_nodes[s].append(nid); stops_here[s].append(l)
        rows = []; cols = []; wts = []
        for l, line in enumerate(routes):
            for i in range(len(line)-1):
                a = node_id[(line[i], l)]; b = node_id[(line[i+1], l)]
                t = self.dur[line[i], line[i+1]]
                rows += [a, b]; cols += [b, a]; wts += [t, t]
        for s, ls in stops_here.items():
            if len(ls) < 2: continue
            for li in ls:
                for lj in ls:
                    if li == lj: continue
                    rows.append(node_id[(s, li)]); cols.append(node_id[(s, lj)])
                    wts.append(self.transfer_penalty + self.perc_wait(lj))
        nbase = len(node_id)
        src_ids = []
        for k, s in enumerate(self.origin_stops):
            src = nbase + k; src_ids.append(src)
            for l in stops_here.get(s, []):
                rows.append(src); cols.append(node_id[(s, l)]); wts.append(self.perc_wait(l))
        nnodes = nbase + len(self.origin_stops)
        csr = csr_matrix((wts, (rows, cols)), shape=(nnodes, nnodes))
        return csr, src_ids, stop_nodes, node_id

    def utvardera(self, routes, return_predecessors=False):
        csr, src_ids, stop_nodes, node_id = self._bygg_graf(routes)
        out = dijkstra(csr, directed=True, indices=src_ids,
                       return_predecessors=return_predecessors)
        dist = out[0] if return_predecessors else out
        stop_dist = np.full((len(self.origin_stops), self.N), np.inf)
        for s, nids in stop_nodes.items():
            stop_dist[:, s] = dist[:, nids].min(axis=1)
        tt = stop_dist[self.o_rows, self.d_arr]
        tt = np.where(self.same, 0.0, tt)
        served = np.isfinite(tt)
        obet_w = self.w_arr[~served].sum(); serv_w = self.w_arr[served].sum()
        serv_cost = (self.w_arr[served]*tt[served]).sum()
        obj = (serv_cost + self.penalty_unserved*obet_w)/self.W
        metrik = {'objektiv': obj, 'obetjanad_andel': obet_w/self.W,
                  'medel_restid_betjanad': serv_cost/max(serv_w, 1e-9)}
        if return_predecessors:
            return obj, metrik, dist, out[1], stop_dist, node_id
        return obj, metrik

In [ ]:
def genetisk_algoritm(central, antal_linjer, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt,
                      utv, headways, pop_size, generationer, mutationsgrad, rng, verbose=True):
    def ny_ind():
        return bygg_individ(central, antal_linjer, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng)
    def ny_linje():
        return bygg_linje(central, lmin, lmax, dur_min, dist_km, grannar, hpl_vikt, rng)
    cache = {}
    def nyckel(ind):
        return tuple(tuple(r) for r in ind)
    def poang(ind):
        k = nyckel(ind)
        if k not in cache:
            cache[k] = utv.utvardera(ind)[0]
        return cache[k]
    pop = [ny_ind() for _ in range(pop_size)]
    historik = []
    for g in range(generationer):
        pop.sort(key=poang)
        basta = poang(pop[0]); historik.append(basta)
        if verbose:
            print(f'  Gen {g+1:3d}/{generationer}: bästa objektiv = {basta:.3f}')
        elit_n = max(2, pop_size//2 - 2)
        elit = pop[:elit_n] + [ny_ind() for _ in range(2)]
        ny_pop = list(elit)
        while len(ny_pop) < pop_size:
            p1, p2 = rng.choice(elit), rng.choice(elit)
            barn = [list(p1[i]) if rng.random() < 0.5 else list(p2[i]) for i in range(antal_linjer)]
            if rng.random() < mutationsgrad:
                nya = []
                for r in barn:
                    if rng.random() < 0.15:            # strukturell mutation: bygg om hela linjen
                        nya.append(ny_linje())
                    else:                               # lokal mutation: justera ändhållplats
                        nya.append(mutera_linje(r, central, lmin, lmax, dist_km, grannar, rng))
                barn = nya
            ny_pop.append(barn)
        pop = ny_pop
    pop.sort(key=poang)
    return pop[0], historik

def _rakna_byten(pred_row, target, id_node):
    """Räknar antal byten (linjebyten) längs den återuppbyggda vägen till target."""
    kedja = []; cur = target; guard = 0
    while cur >= 0 and guard < 1000000:
        kedja.append(cur); nxt = pred_row[cur]
        if nxt == cur:
            break
        cur = nxt; guard += 1
    kedja = kedja[::-1]
    byten = 0
    for i in range(len(kedja)-1):
        a = id_node.get(kedja[i]); b = id_node.get(kedja[i+1])
        if a and b and a[0] == b[0] and a[1] != b[1]:
            byten += 1
    return byten

def analysera_basta(routes, utv):
    """Detaljerad analys av bästa lösningen inkl. bytesfördelning (via predecessors).
    Returnerar (obj, metrik, byten_hist) där byten_hist mappar antal_byten -> efterfrågan.
    Nyckeln -1 = obetjänad efterfrågan."""
    obj, metrik, dist, pred, stop_dist, node_id = utv.utvardera(routes, return_predecessors=True)
    id_node = {v: k for k, v in node_id.items()}  # nid -> (stop, line)
    origin_row = {s: i for i, s in enumerate(utv.origin_stops)}
    stop_nodes = defaultdict(list)
    for (s, l), nid in node_id.items():
        stop_nodes[s].append(nid)
    byten_hist = defaultdict(float)  # antal_byten -> efterfrågan (-1 = obetjänad)
    for o, d, w, same in zip(utv.o_arr, utv.d_arr, utv.w_arr, utv.same):
        if same:
            byten_hist[0] += w; continue
        r = origin_row[o]
        nids = stop_nodes.get(d, [])
        if not nids:
            byten_hist[-1] += w; continue
        # välj den slutnod som faktiskt ger kortaste distansen
        target = min(nids, key=lambda n: dist[r, n])
        if not np.isfinite(dist[r, target]):
            byten_hist[-1] += w; continue
        byten_hist[_rakna_byten(pred[r], target, id_node)] += w
    return obj, metrik, byten_hist

## 8. Kör den genetiska algoritmen

In [ ]:
# headways: skalär => samma för alla linjer; lista => per linje
if isinstance(HEADWAY_MIN, (list, tuple)):
    assert len(HEADWAY_MIN) == ANTAL_LINJER, 'HEADWAY_MIN-listan måste ha längd ANTAL_LINJER'
    headways = [float(h) for h in HEADWAY_MIN]
else:
    headways = [float(HEADWAY_MIN)] * ANTAL_LINJER

grannar = bygg_grannar(dur_min, ANTAL_GRANNAR)
hpl_vikt = demand_per_hpl(hpl_od, N)   # total efterfrågan per hållplats (för kartan)

# Ankarvikt = efterfrågan × avstånd från centralen. Det får linjearmarna att sträva UTÅT mot
# resenärer längre bort (annars fastnar linjerna i den täta Mariehamnskärnan där efterfrågan är störst).
avstand_fran_central = dist_km[central].copy()
ankar_vikt = hpl_vikt * avstand_fran_central

utv = Utvarderare(N, dur_min, hpl_od, headways,
                  transfer_penalty=BYTESSTRAFF_MIN, wait_weight=VANTETIDSVIKT,
                  penalty_unserved=STRAFF_OBETJANAD)

t0 = time.time()
basta, historik = genetisk_algoritm(
    central, ANTAL_LINJER, LINJE_LANGD_MIN_KM, LINJE_LANGD_MAX_KM,
    dur_min, dist_km, grannar, ankar_vikt, utv, headways,
    POP_STORLEK, GENERATIONER, MUTATIONSGRAD, rng, verbose=False)
print(f'Klart på {time.time()-t0:.1f} s. Bästa objektiv: {historik[0]:.2f} -> {historik[-1]:.2f}')

## 9. Resultat och nyckeltal

In [ ]:
obj, metrik, byten_hist = analysera_basta(basta, utv)
tot_w = sum(byten_hist.values())

print('=== Nyckeltal för bästa linjenät ===')
print(f'Objektiv (upplevd medelrestid inkl. straff): {obj:.2f} min')
print(f'Efterfrågeviktad medelrestid (betjänad):     {metrik["medel_restid_betjanad"]:.1f} min')
print(f'Andel obetjänad efterfrågan:                 {metrik["obetjanad_andel"]*100:.1f} %')
print()
print('Bytesfördelning (andel av efterfrågan):')
for k in sorted(byten_hist):
    etikett = 'obetjänad' if k == -1 else f'{k} byten'
    print(f'  {etikett:>12}: {byten_hist[k]/tot_w*100:5.1f} %')

print('\n=== Linjer ===')
rader = []
for i, linje in enumerate(basta):
    langd = linjelangd(linje, dist_km)
    rader.append({'linje': i+1, 'antal_hpl': len(linje), 'langd_km': round(langd, 1),
                  'headway_min': headways[i],
                  'fran': hpl.loc[linje[0], 'namn'], 'till': hpl.loc[linje[-1], 'namn']})
linjer_df = pd.DataFrame(rader)
print(f'Total linjelängd: {linjer_df["langd_km"].sum():.1f} km')
linjer_df

## 10. Visualisering

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(historik)+1), historik, marker='o', ms=3)
plt.xlabel('Generation'); plt.ylabel('Bästa objektiv (upplevd restid, min)')
plt.title('GA-konvergens'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))

# alla hållplatser, storlek efter efterfrågan
if hpl_vikt.max() > 0:
    storlek = 6 + 350 * (hpl_vikt / hpl_vikt.max())
else:
    storlek = np.full(N, 10.0)
ax.scatter(hpl['lon'], hpl['lat'], s=storlek, c='lightgray', edgecolors='none', zorder=1)

farger = plt.cm.tab10(np.linspace(0, 1, max(ANTAL_LINJER, 1)))
for i, linje in enumerate(basta):
    lons = hpl.loc[linje, 'lon'].to_numpy(); lats = hpl.loc[linje, 'lat'].to_numpy()
    ax.plot(lons, lats, '-', color=farger[i], lw=2, alpha=0.85,
            label=f'Linje {i+1} ({linjelangd(linje, dist_km):.0f} km)')
    ax.scatter(lons, lats, s=12, color=farger[i], zorder=3)

ax.scatter([hpl.loc[central, 'lon']], [hpl.loc[central, 'lat']],
           marker='*', s=500, color='black', zorder=5, label='Central hållplats')
ax.set_aspect(1/np.cos(np.radians(float(hpl['lat'].mean()))))
ax.set_title('Optimerat busslinjenät – Åland (EM-rusning)')
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
plt.tight_layout(); plt.savefig(PNG_KARTA, dpi=130); plt.show()
print(f'Karta sparad: {PNG_KARTA}')

## 11. Export av bästa lösningen

In [ ]:
rader = []
for i, linje in enumerate(basta):
    for ordn, s in enumerate(linje):
        rader.append({'linje': i+1, 'ordning': ordn, 'hpl_id': hpl.loc[s, 'hpl_id'],
                      'namn': hpl.loc[s, 'namn'], 'lon': hpl.loc[s, 'lon'], 'lat': hpl.loc[s, 'lat']})
resultat_df = pd.DataFrame(rader)
resultat_df.to_csv(CSV_RESULTAT, index=False)
print(f'Bästa linjenät sparat: {CSV_RESULTAT}  ({len(resultat_df)} rader)')
resultat_df.head(12)